# 形状变换

宽表和长表不是两种数据，而是同一份数据的两种摆法：画热力图、喂给机器学习模型时希望是宽表（每个类别一列），做分组统计、画时间序列时又希望是长表（类别名单独一列）。Polars 为此准备了一组互逆的方法。

这一章依次看 `pivot` / `unpivot`、`unstack`、`to_dummies`：前两个在长宽之间来回转换，`unstack` 按固定步长把值摊成多列，`to_dummies` 把类别展开成 0/1 列。

In [1]:
import polars as pl
from helper.jupyter import row

## 透视表

透视表是把行和列对调、重新整理聚合结果的表，可以理解成把「长表（long）」变成「宽表（wide）」：`pivot()` 走这个方向，`unpivot()` 把宽表退回长表。

两者并不是简单的转置，因为同一个行、列组合可能对应多个值，必须由聚合函数决定留哪一个。

### pivot

`pivot()` 把指定列的值变成新的列名，用另一列当行键，剩下的列作为单元格内容重新摆放。

下面的例子用 `name` 做列标签、`index` 做行标签、`value` 做单元格的值；同一行同一列出现多个值时，用 `aggregate_function` 指定的方式（这里是 `first`）合成一个。可以看到行数没变，只是排成了新形状。

In [2]:
df = pl.DataFrame(
    dict(
        index=[0, 1, 2, 0, 1, 2, 0, 1, 2, 1],
        name=["A", "A", "A", "B", "B", "B", "C", "C", "C", "A"],
        value=[10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
        score=[1, 3, 5, 4, 3, 2, 1, 2, 3, 4],
    )
)

# 用 name 做列、index 做行，把 value 摆进去（重复时取第一个）
df1 = df.pivot("name", index="index", values="value", aggregate_function="first")
row(df, df1)

index,name,value,score
i64,str,i64,i64
0,"""A""",10,1
1,"""A""",20,3
2,"""A""",30,5
0,"""B""",40,4
1,"""B""",50,3
2,"""B""",60,2
0,"""C""",70,1
1,"""C""",80,2
2,"""C""",90,3


`aggregate_function` 决定怎么合并重复的格子，常用的有三种。

1. **`sum`** — 求和，适合「同一格多次记录要累加」的场景
2. **`mean`** — 求平均，适合需要削弱重复记录影响的场景
3. **任意表达式** — 传 `pl.element()` 就把匹配的值收成一个列表，原样保留

下面把三种结果并排比一下。注意结果列的类型也跟着变：`sum` 和 `mean` 得到数值，`pl.element()` 得到列表。

In [3]:
# 同一组合的值分别按「求和」「平均」「列表」来合并
df2 = df.pivot("name", index="index", values="value", aggregate_function="sum")
df3 = df.pivot("name", index="index", values="value", aggregate_function="mean")
df4 = df.pivot("name", index="index", values="value", aggregate_function=pl.element())
row(df2, df3, df4)

index,A,B,C
i64,i64,i64,i64
0,10,40,70
1,120,50,80
2,30,60,90
index,A,B,C
i64,f64,f64,f64
0,10.0,40.0,70.0
1,60.0,50.0,80.0
2,30.0,60.0,90.0
index,A,B,C


`values` 传多列时，每列都会展开，结果列名是 `<值列名><separator><透视列的值>`；`separator` 可以改分隔符，默认是下划线。

列名一长就容易看混，所以在多维透视时换一个显眼的分隔符是常见做法。

In [4]:
# 同时展开 value 和 score 两列
df5 = df.pivot("name", index="index", values=["value", "score"], aggregate_function=pl.element())
df6 = df.pivot("name", index="index", values=["value", "score"], aggregate_function="max", separator=":")
row(df5, df6)

index,value_A,value_B,value_C,score_A,score_B,score_C
i64,list[i64],list[i64],list[i64],list[i64],list[i64],list[i64]
0,[10],[40],[70],[1],[4],[1]
1,"[20, 100]",[50],[80],"[3, 4]",[3],[2]
2,[30],[60],[90],[5],[2],[3]
index,value:A,value:B,value:C,score:A,score:B,score:C
i64,i64,i64,i64,i64,i64,i64
0,10,40,70,1,4,1
1,100,50,80,4,3,2
2,30,60,90,5,2,3


### unpivot

`unpivot()` 是 `pivot()` 的逆操作，把宽表退回长表：`on` 指定的列会被叠成一列，`index` 指定的列原样保留。

- **`on`** — 要叠起来的列。可以给列名、列名列表或选择器；要排除键列时用 `cs.exclude()` 最省事，表达式（`pl.col()` 之类）则不能传。
- **`index`** — 原样保留的列。
- **`variable_name`** — 存放原列名的新列名。
- **`value_name`** — 存放值的新列名。

下面的例子把 `index` 以外的列都叠起来，原列名放进 `name`，值放进 `value`。

In [5]:
from polars import selectors as cs

# 把 index 以外的列叠成长表
df_u1 = df1.unpivot(on=cs.exclude("index"), index="index", variable_name="name", value_name="value")
row(df1, df_u1)

shape: (3, 4) index A B C i64 i64 i64 i64 0 10 40 70 1 20 50 80 2 30 60 90 
 shape: (9, 3) index name value i64 str i64 0 "A" 10 1 "A" 20 2 "A" 30 0 "B" 40 1 "B" 50 2 "B" 60 0 "C" 70 1 "C" 80 2 "C" 90

`values` 传多列做出来的 `df6`，要还原就得先把列名拆开再 pivot 回去，因为原信息被压进了 `value:A` 这样的列名里。步骤是：

1. **`unpivot(on=cs.exclude('index'), index='index')`** — 先退回长表，`variable` 列里是 `value:A`、`score:B` 这样的原列名。
2. **`pl.col('variable').str.split(':')`** — 按 `:` 切开，得到 `['value', 'A']`。
3. **`.list.to_struct(fields=['colname', 'name'])`** — 转成两个字段的结构体。
4. **`.struct.unnest()`** — 展开成 `colname` 和 `name` 两列。
5. **`pivot('colname', index=['index', 'name'], values='value')`** — 用原列名 pivot 回去，就还原了。

这一串看着长，但每一步只做一件事，出了问题也容易定位是哪一步形状不对。

In [6]:
(
    df6.unpivot(on=cs.exclude("index"), index="index")
    .with_columns(
        pl.col("variable")
        .str.split(":")  # value:A → ['value', 'A']
        .list.to_struct(fields=["colname", "name"])
        .struct.unnest()  # 拆成两列
    )
    .pivot("colname", index=["index", "name"], values="value")
)

index,name,value,score
i64,str,i64,i64
0,"""A""",10,1
1,"""A""",100,4
2,"""A""",30,5
0,"""B""",40,4
1,"""B""",50,3
2,"""B""",60,2
0,"""C""",70,1
1,"""C""",80,2
2,"""C""",90,3


## unstack

`unstack()` 接近 NumPy 的 `reshape`：它不按值聚合，而是按固定步长把列里的值摊成多列。`step` 是每组的行数，`how` 是展开方向（`'vertical'` 行优先，`'horizontal'` 列优先），不够的位置用 `fill_values` 填。

Polars 2.0 里这些参数全是关键字专用，所以必须写成 `df.unstack(step=4, ...)`，传位置参数会直接报错。

In [7]:
# 每 4 行纵向展开一次（不够的格子用 -1 和 '?' 填）
df_vertical = df.unstack(step=4, how="vertical", columns=["index", "name"], fill_values=[-1, "?"])
# 每 3 行横向展开一次
df_horizontal = df.unstack(step=3, how="horizontal", columns="index")
row(df, df_vertical, df_horizontal)

index,name,value,score
i64,str,i64,i64
0,"""A""",10,1
1,"""A""",20,3
2,"""A""",30,5
0,"""B""",40,4
1,"""B""",50,3
2,"""B""",60,2
0,"""C""",70,1
1,"""C""",80,2
2,"""C""",90,3


`to_dummies()` 把类别列展开成 0/1 的哑变量（one-hot 编码）：命中为 1，否则为 0。给多列就每列各自展开，列名是「原列名_类别值」。

- `drop_first=False`（默认）— 保留所有类别。
- `drop_first=True` — 去掉第一个类别作为基准，做回归时用来避免多重共线性。

In [8]:
# 把 index 和 name 展开成哑变量
df_original = df.filter(pl.col("index") < 2).select("index", "name", "value")
df_dummies1 = df_original.to_dummies(columns=["index", "name"], drop_first=False)
df_dummies2 = df_original.to_dummies(columns=["index", "name"], drop_first=True)
row(df_original, df_dummies1, df_dummies2)

shape: (7, 3) index name value i64 str i64 0 "A" 10 1 "A" 20 0 "B" 40 1 "B" 50 0 "C" 70 1 "C" 80 1 "A" 100 
 shape: (7, 6) index_0 index_1 name_A name_B name_C value u8 u8 u8 u8 u8 i64 1 0 1 0 0 10 0 1 1 0 0 20 1 0 0 1 0 40 0 1 0 1 0 50 1 0 0 0 1 70 0 1 0 0 1 80 0 1 1 0 0 100 
 shape: (7, 4) index_1 name_B name_C value u8 u8 u8 i64 0 0 0 10 1 0 0 20 0 1 0 40 1 1 0 50 0 0 1 70 1 0 1 80 1 0 0 100